- Bronze -> Silver for `external_payment` (payment-system .dat feed, 15 rows total across 3 days).

In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "external_payment"    # kept distinct from internal `payment` (payment.csv) - different grain/source
BRONZE_SOURCE_NAME = "external_payment"
business_date_str = date.today().isoformat()

APPROVED_STATUSES = ["INITIATED", "SETTLED", "FAILED", "REVERSED"]

In [0]:
bronze_df = read_bronze(spark, BRONZE_SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

Bronze row count: 15
root
 |-- payment_id: string (nullable = true)
 |-- fund_id: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- settlement_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _source_file_record_count_mismatch: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



### 1. Type casting
`business_date` is derived from `event_timestamp` (Bronze has no
pre-extracted date column here, same as `external_position` /
`external_cash` / `external_reference`).

In [0]:
typed_df = (
    bronze_df
    .withColumn("payment_id", F.trim(F.col("payment_id")))
    .withColumn("fund_id", F.trim(F.col("fund_id")))            # external code, e.g. FND001
    .withColumn("payment_type", F.upper(F.trim(F.col("payment_type"))))
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("settlement_timestamp", F.to_timestamp("settlement_timestamp"))
    .withColumn("source_system", F.trim(F.col("source_system")))
    .withColumn("business_date", F.to_date(F.col("event_timestamp")))
)

### 2. Status validity check
Not a hard reject (Bronze already validated core fields) - just
visibility if an unexpected status value shows up.

In [0]:
bad_status_df = typed_df.filter(~F.col("status").isin(APPROVED_STATUSES))
bad_status_count = bad_status_df.count()
if bad_status_count > 0:
    write_quarantine(bad_status_df.withColumn("reason_code", F.lit("INVALID_STATUS")), SOURCE_NAME)
    print(f"WARNING: {bad_status_count} rows with unrecognized status.")

### 3. Crosswalk join

In [0]:
fund_xwalk = read_crosswalk(spark, "fund")

FUND_XWALK_EXTERNAL_COL = "external_fund_id"
FUND_XWALK_INTERNAL_COL = "internal_fund_id"

fund_xwalk_slim = fund_xwalk.select(
    F.col(FUND_XWALK_EXTERNAL_COL).alias("fund_id"),
    F.col(FUND_XWALK_INTERNAL_COL).alias("internal_fund_id"),
)

joined_df = typed_df.join(fund_xwalk_slim, on="fund_id", how="left")

xwalk_miss_df = joined_df.filter(F.col("internal_fund_id").isNull()) \
    .withColumn("reason_code", F.lit("UNKNOWN_CROSSWALK_MAPPING"))
xwalk_miss_count = xwalk_miss_df.count()
if xwalk_miss_count > 0:
    write_quarantine(xwalk_miss_df, SOURCE_NAME)
    print(f"WARNING: {xwalk_miss_count} rows failed crosswalk lookup.")

crosswalked_df = joined_df.filter(F.col("internal_fund_id").isNotNull())

### 4. Exact duplicate detection
Business key = payment_id . 

In [0]:
KEY_COLS = ["payment_id"]
COMPARE_COLS = ["internal_fund_id", "payment_type", "amount", "status"]

deduped_df, duplicates_df, breaks_df = split_duplicates(crosswalked_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
break_count = breaks_df.count()   # same event id, different values - shouldn't happen, but flagged if it does

if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("PAYMENT_EVENT_BREAK")), SOURCE_NAME)

### 5. Write to Silver


In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")
print("Status breakdown:")
deduped_df.groupBy("status").count().show()

Silver row count: 15
Status breakdown:
+---------+-----+
|   status|count|
+---------+-----+
|  SETTLED|    8|
|INITIATED|    3|
|   FAILED|    3|
| REVERSED|    1|
+---------+-----+



In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "invalid_status", typed_df.count(), bad_status_count, "INVALID_STATUS")
log_dq(spark, SOURCE_NAME, business_date_str, "crosswalk_miss", joined_df.count(), xwalk_miss_count, "UNKNOWN_CROSSWALK_MAPPING")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", crosswalked_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, SOURCE_NAME, business_date_str, "payment_event_break", crosswalked_df.count(), break_count, "PAYMENT_EVENT_BREAK")

/home/spark-5e029a95-92d0-40c0-9f81-ca/.ipykernel/71/command-5696143635338729-2886423099:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
# bad_status rows are flagged for visibility but NOT removed (see Step 2) - not counted here
quarantined_count = xwalk_miss_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=15 = silver=15 + quarantined=0
